# Experiment: Origen y adquisición del corpus Doppler

**Pregunta.** ¿De qué fuentes sale el audio crudo, con qué licencias y cómo se alinea cada corpus con las dos tareas (detección binaria vs tipo de sirena)?

**Criterio de éxito.** Inventario reproducible (local y Kaggle), tabla tarea↔dataset, y CSV de archivos detectados en `reports/tables/`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

SEED = 7

def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/working"),
        Path("/kaggle/input/doppler-ml"),
    ]
    for parent in Path.cwd().resolve().parents:
        candidates.append(parent)
    for cand in candidates:
        if (cand / "src" / "paths.py").exists():
            return cand
    return Path.cwd()

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import corpus_paths, figures_dir, is_kaggle, tables_dir

print("repo:", REPO_ROOT)
print("kaggle:", is_kaggle())
print("available:", list(corpus_paths().available()))
SEED


repo: /home/jeancdevx/dev/doppler/doppler-ml
kaggle: False
available: ['sirennet', 'lssiren', 'urbansound8k']


7

## Plan

- Hipótesis: ningún dataset único cubre tipo (4 clases) + in-the-wild + distractores urbanos.
- Barrido: sireNNet, LSSiren, UrbanSound8K (trabajo) y AudioSet-EV v2 (escala, no descargado aquí).
- Métricas de esta notebook: n_archivos, clases, presencia/ausencia de WAV, licencia.


In [ ]:
import pandas as pd

FIG = figures_dir()
TAB = tables_dir()

catalog = pd.DataFrame([
    {
        "corpus": "sireNNet",
        "role": "primario_4clases",
        "task": "binary+multiclass",
        "n_nominal": 1675,
        "classes": "ambulance, police, firetruck, traffic",
        "license": "CC BY 4.0",
        "source": "https://data.mendeley.com/datasets/j4ydzzv4kb/1",
        "citation": "Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",
        "notes": "Release publico ya aumentado; no hay split original vs augment",
    },
    {
        "corpus": "LSSiren",
        "role": "binario_in_the_wild",
        "task": "binary",
        "n_nominal": 1800,
        "classes": "siren, road_noise",
        "license": "CC BY 4.0",
        "source": "https://doi.org/10.6084/m9.figshare.19291472",
        "citation": "Asif et al., Sci Data 2022, 10.1038/s41597-022-01727-2",
        "notes": "Karachi + setup experimental + internet; sesgo a ambulancia",
    },
    {
        "corpus": "UrbanSound8K",
        "role": "distractores_urbanos",
        "task": "urban_scene / binary_generic_siren",
        "n_nominal": 8732,
        "classes": "10 clases urbanas incl. siren (~929)",
        "license": "CC BY-NC 4.0",
        "source": "https://zenodo.org/records/1203745",
        "citation": "Salamon, Jacoby & Bello, 2014, 10.5281/zenodo.1203745",
        "notes": "Usar folds oficiales; no partir el mismo fsID",
    },
    {
        "corpus": "AudioSet-EV v2",
        "role": "escala_entrenamiento_posterior",
        "task": "binary+multiclass",
        "n_nominal": 28816,
        "classes": "police / ambulance / fire vs urban negatives",
        "license": "subset AudioSet (YouTube); uso de investigacion",
        "source": "https://doi.org/10.5281/zenodo.18668076",
        "citation": "Giacomelli & Rinaldi, 2025, 10.5281/zenodo.18668076",
        "notes": "~8-16 GB zip / ~28 GB WAV; no se descarga en esta fase",
    },
])
catalog.to_csv(TAB / "corpus_catalog.csv", index=False)
catalog


,corpus,role,task,n_nominal,classes,license,source,citation,notes
0,sireNNet,primario_4clases,binary+multiclass,1675,"ambulance, police, firetruck, traffic",CC BY 4.0,https://data.mendeley.com/datasets/j4ydzzv4kb/1,"Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",Release publico ya aumentado; no hay split ori...
1,LSSiren,binario_in_the_wild,binary,1800,"siren, road_noise",CC BY 4.0,https://doi.org/10.6084/m9.figshare.19291472,"Asif et al., Sci Data 2022, 10.1038/s41597-022...",Karachi + setup experimental + internet; sesgo...
2,UrbanSound8K,distractores_urbanos,urban_scene / binary_generic_siren,8732,10 clases urbanas incl. siren (~929),CC BY-NC 4.0,https://zenodo.org/records/1203745,"Salamon, Jacoby & Bello, 2014, 10.5281/zenodo....",Usar folds oficiales; no partir el mismo fsID
3,AudioSet-EV v2,escala_entrenamiento_posterior,binary+multiclass,28816,police / ambulance / fire vs urban negatives,subset AudioSet (YouTube); uso de investigacion,https://doi.org/10.5281/zenodo.18668076,"Giacomelli & Rinaldi, 2025, 10.5281/zenodo.186...",~8-16 GB zip / ~28 GB WAV; no se descarga en e...


## Alineación tarea ↔ corpus

El protocolo de splits evita mezclar orígenes en el mismo fold de entrenamiento sin declararlo.


In [ ]:
alignment = pd.DataFrame([
    {"task": "deteccion_binaria", "train": "sireNNet (sirena vs traffic)", "test_internal": "holdout sireNNet", "test_external": "LSSiren; UrbanSound8K siren vs resto"},
    {"task": "tipo_multiclase", "train": "sireNNet (ambulance/police/firetruck)", "test_internal": "holdout sireNNet agrupado", "test_external": "AudioSet-EV eval (fase posterior)"},
    {"task": "distractores_duros", "train": "UrbanSound8K no-siren (folds 1-8)", "test_internal": "folds 9-10", "test_external": "LSSiren road_noise"},
])
alignment.to_csv(TAB / "task_alignment.csv", index=False)
alignment


,task,train,test_internal,test_external
0,deteccion_binaria,sireNNet (sirena vs traffic),holdout sireNNet,LSSiren; UrbanSound8K siren vs resto
1,tipo_multiclase,sireNNet (ambulance/police/firetruck),holdout sireNNet agrupado,AudioSet-EV eval (fase posterior)
2,distractores_duros,UrbanSound8K no-siren (folds 1-8),folds 9-10,LSSiren road_noise


## Inventario de archivos montados

Si un corpus no está, la notebook no falla: registra `missing` y sigue.


In [ ]:
from src.inventory import scan_lssiren, scan_sirennet, scan_urbansound8k

paths = corpus_paths()
frames = []
status_rows = []

scanners = {
    "sirennet": (paths.sirennet, scan_sirennet),
    "lssiren": (paths.lssiren, scan_lssiren),
    "urbansound8k": (paths.urbansound8k, scan_urbansound8k),
}

for name, (root, scanner) in scanners.items():
    if root is None:
        status_rows.append({"corpus": name, "status": "missing", "root": None, "n_rows": 0})
        continue
    df = scanner(root)
    frames.append(df)
    status_rows.append({"corpus": name, "status": "ok", "root": str(root), "n_rows": int(len(df))})

status = pd.DataFrame(status_rows)
status.to_csv(TAB / "mount_status.csv", index=False)
inventory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["corpus", "path", "relpath", "label", "task"])
if not inventory.empty:
    inventory.to_csv(TAB / "file_inventory.csv", index=False)
status


,corpus,status,root,n_rows
0,sirennet,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1675
1,lssiren,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1834
2,urbansound8k,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,8732


In [5]:
if inventory.empty:
    counts = pd.DataFrame(columns=["corpus", "label", "n"])
else:
    counts = inventory.groupby(["corpus", "label"]).size().reset_index(name="n")
counts.to_csv(TAB / "label_counts.csv", index=False)
counts


,corpus,label,n
0,lssiren,road_noise,902
1,lssiren,siren,932
2,sirennet,ambulance,400
3,sirennet,firetruck,400
4,sirennet,police,454
5,sirennet,traffic,421
6,urbansound8k,air_conditioner,1000
7,urbansound8k,car_horn,429
8,urbansound8k,children_playing,1000
9,urbansound8k,dog_bark,1000


## Resultados

- El catálogo y el protocolo de splits quedan fijados antes de entrenar.
- `file_inventory.csv` es la tabla semiestructurada que alimenta las notebooks 02 y 03.
- Siguiente: exploración acústica (duración, sample rate, formas de onda).


In [ ]:
result = {
    "seed": SEED,
    "n_catalog_rows": int(len(catalog)),
    "mounted": status.set_index("corpus")["status"].to_dict(),
    "n_inventory_rows": int(len(inventory)),
    "tables": [p.name for p in sorted(TAB.glob("*.csv"))],
}
result


{'seed': 7,
 'n_catalog_rows': 4,
 'mounted': {'sirennet': 'ok', 'lssiren': 'ok', 'urbansound8k': 'ok'},
 'n_inventory_rows': 12241,
 'tables': ['corpus_catalog.csv',
  'file_inventory.csv',
  'label_counts.csv',
  'mount_status.csv',
  'task_alignment.csv']}